# Доп. задание: обзор open-source решений для Face Recognition

Задание (3 балла): отобрать 3 актуальные open-source библиотеки для face recognition, сделать
подробный отчёт по каждой (возможности, зоопарк моделей, сложности установки, требования к ресурсам,
собственное мнение), с примерами кода, протестированными на своих изображениях.

Отобраны три библиотеки, которые чаще всего называют опорными для практического face recognition на
2026 год: **InsightFace**, **DeepFace** и **facenet-pytorch**. Первые две — по разным причинам самые
цитируемые и используемые в индустрии решения, третья — самая простая точка входа именно на PyTorch
(том же фреймворке, что использовался во всём остальном проекте).

**Тестовые фото** берутся из уже подготовленного датасета Task 2 (`data/prepared_recog/images/`) —
подключите Output вашего Task 2 ноутбука через `+ Add Data`, путь поправьте в первой ячейке ниже.


In [3]:
import os

# Поправьте под свой реальный slug Task 2 ноутбука (проверить: os.listdir('/kaggle/input/notebooks/<юзернейм>'))
TASK2_INPUT_DIR = '/kaggle/input/notebooks/iuliiaburmistrova/notebook98e2177099'

RECOG_IMAGES_DIR = os.path.join(TASK2_INPUT_DIR, 'data', 'prepared_recog', 'images')
RECOG_CSV = os.path.join(TASK2_INPUT_DIR, 'data', 'prepared_recog', 'selected_dataset.csv')

import pandas as pd

recog_df = pd.read_csv(RECOG_CSV)
counts = recog_df['identity'].value_counts()

# та же пара, что использовалась в Task 3 для демонстрации (тот же человек / другой человек)
same_person_identity = counts[counts >= 2].index[0]
same_person_photos = recog_df[recog_df['identity'] == same_person_identity]['image_id'].tolist()[:2]
other_identity = counts.index[counts.index != same_person_identity][0]
other_person_photo = recog_df[recog_df['identity'] == other_identity]['image_id'].tolist()[0]

image_path_1 = os.path.join(RECOG_IMAGES_DIR, same_person_photos[0])
image_path_2 = os.path.join(RECOG_IMAGES_DIR, same_person_photos[1])   # тот же человек, что и image_path_1
image_path_3 = os.path.join(RECOG_IMAGES_DIR, other_person_photo)      # другой человек

print('Фото 1 (identity=%s):' % same_person_identity, image_path_1)
print('Фото 2 (тот же человек):', image_path_2)
print('Фото 3 (другой человек, identity=%s):' % other_identity, image_path_3)


Фото 1 (identity=9240): /kaggle/input/notebooks/iuliiaburmistrova/notebook98e2177099/data/prepared_recog/images/144048.jpg
Фото 2 (тот же человек): /kaggle/input/notebooks/iuliiaburmistrova/notebook98e2177099/data/prepared_recog/images/082666.jpg
Фото 3 (другой человек, identity=9114): /kaggle/input/notebooks/iuliiaburmistrova/notebook98e2177099/data/prepared_recog/images/132832.jpg


## 1. InsightFace

**GitHub:** [deepinsight/insightface](https://github.com/deepinsight/insightface)

### Возможности и преимущества

InsightFace — открытый набор инструментов для 2D и 3D анализа лиц, построенный в основном на PyTorch
и MXNet. Библиотека реализует полный набор задач: детекцию лиц, распознавание, выравнивание, а также
более специфичные вещи вроде face swap. Именно в InsightFace впервые опубликована референсная
реализация ArcFace — той самой angular margin loss, которую мы реализовывали в Task 2 своими руками, и
большинство современных SOTA-моделей face recognition обучаются на ArcFace или его производных
(например, AdaFace с адаптивным margin).

### Зоопарк моделей

Библиотека предоставляет несколько семейств backbone'ов для распознавания — IResNet, MobileFaceNet,
MobileNet, InceptionResNet_v2, DenseNet и другие, обученные с разными loss-функциями (в первую очередь
ArcFace и его варианты, включая Sub-center ArcFace). Для детекции — модели семейства RetinaFace/SCRFD.
Готовые предобученные веса (buffalo_l и другие "packs") можно скачать одной командой через
`insightface.app.FaceAnalysis`.

### Сложности установки

Основная боль — MXNet-зависимости в старых версиях и необходимость собирать C++ расширения. По данным
разработчиков, в 2026 году вышла версия InsightFace 1.0 с более лёгкой установкой на Python, без
обязательной сборки C++ (упрощение специально сделано для случаев, когда C++ toolchain недоступен).
Тем не менее для GPU-инференса нужен `onnxruntime-gpu`, версия которого должна точно соответствовать
версии CUDA — рассинхронизация версий CUDA/onnxruntime — самая частая причина проблем при установке.

### Требования к ресурсам

Для инференса достаточно любой GPU с 2-4 ГБ VRAM (модели маленькие, ~100 МБ); для дообучения тяжёлых
backbone'ов на своих данных желателен GPU с 16+ ГБ VRAM. CPU-инференс тоже работает через ONNX Runtime,
но заметно медленнее.

### Личное мнение

InsightFace — самый "исследовательский" и мощный вариант из трёх: если нужна максимальная точность и
готовность разбираться с деталями (несовпадающие версии зависимостей, разные форматы весов) — это
разумный выбор. Для быстрого прототипа или учебного проекта старт может быть более трудоёмким, чем у
DeepFace.


In [7]:
!pip install -q insightface onnxruntime

import insightface
from insightface.app import FaceAnalysis
import cv2
import matplotlib.pyplot as plt
import numpy as np

# buffalo_l - готовый пакет детекция+recognition моделей, скачивается автоматически при первом запуске
app = FaceAnalysis(name='buffalo_l')
app.prepare(ctx_id=0, det_size=(640, 640))  # ctx_id=-1 для CPU

img1 = cv2.imread(image_path_1)
img2 = cv2.imread(image_path_2)  # тот же человек
img3 = cv2.imread(image_path_3)  # другой человек

faces1 = app.get(img1)
faces2 = app.get(img2)
faces3 = app.get(img3)

if faces1 and faces2 and faces3:
    emb1, emb2, emb3 = faces1[0].normed_embedding, faces2[0].normed_embedding, faces3[0].normed_embedding
    sim_same = float(np.dot(emb1, emb2))
    sim_diff = float(np.dot(emb1, emb3))
    print(f'InsightFace cosine similarity (тот же человек): {sim_same:.3f}')
    print(f'InsightFace cosine similarity (другой человек): {sim_diff:.3f}')
else:
    print('Лицо не найдено на одном из фото')


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/w600k_r50.onnx recognition ['None', 3, 112, 112] 127.5 127.5
set det-size: (640, 640)
InsightFace 

## 2. DeepFace

**GitHub:** [serengil/deepface](https://github.com/serengil/deepface)

### Возможности и преимущества

DeepFace — лёгкий "фреймворк-обёртка", который прячет за собой сразу несколько моделей и берёт на себя
весь пайплайн (детекция → выравнивание → нормализация → извлечение эмбеддинга → сравнение) в одну
строчку кода. Помимо распознавания, из коробки есть анализ атрибутов лица (возраст, пол, эмоция, раса).
По заявлению авторов, экспериментально показанная точность используемых моделей на задачах
верификации лиц превышает точность человека (у людей она обычно оценивается на уровне около 97.5%).

### Зоопарк моделей

DeepFace оборачивает целый набор готовых моделей на выбор: VGG-Face, FaceNet, FaceNet512, OpenFace,
DeepFace (модель Facebook, давшая имя всей библиотеке), DeepID, ArcFace, Dlib, SFace, GhostFaceNet,
Buffalo_L (тот же backbone, что и в InsightFace). По умолчанию используется VGG-Face, но модель легко
переключить одним параметром.

### Сложности установки

Самая простая из трёх — устанавливается одной командой `pip install deepface`, без C++ сборки и без
привязки к конкретной версии CUDA (внутри использует TensorFlow/Keras backend). Основная сложность —
именно из-за зависимости от TensorFlow версии иногда конфликтуют с другими установленными в окружении
пакетами (например, если в проекте уже используется PyTorch с определённой версией CUDA).

### Требования к ресурсам

Самая лёгкая по требованиям библиотека из трёх — большинство моделей (особенно OpenFace, SFace) заточены
под быстрый CPU-инференс, GPU не обязателен. Для VGG-Face/FaceNet512 GPU ускоряет заметно, но не
критичен для единичных сравнений лиц.

### Личное мнение

DeepFace — лучший выбор именно для учебного/демонстрационного проекта или быстрого MVP: минимум кода,
минимум мороки с зависимостями, при этом можно быстро сравнить сразу несколько моделей на одних и тех
же фото просто сменив параметр `model_name`.


In [8]:
!pip install -q deepface

from deepface import DeepFace

# Пара "тот же человек"
result_same = DeepFace.verify(img1_path=image_path_1, img2_path=image_path_2, model_name='ArcFace')
print(f"Тот же человек: verified={result_same['verified']}, distance={result_same['distance']:.3f}, "
      f"threshold={result_same['threshold']:.3f}")

# Пара "разные люди"
result_diff = DeepFace.verify(img1_path=image_path_1, img2_path=image_path_3, model_name='ArcFace')
print(f"Разные люди: verified={result_diff['verified']}, distance={result_diff['distance']:.3f}, "
      f"threshold={result_diff['threshold']:.3f}")

# Быстрое сравнение нескольких моделей на паре "тот же человек"
for model_name in ['VGG-Face', 'Facenet512', 'ArcFace', 'SFace']:
    r = DeepFace.verify(img1_path=image_path_1, img2_path=image_path_2, model_name=model_name)
    print(f"{model_name}: verified={r['verified']}, distance={r['distance']:.3f}")


Тот же человек: verified=False, distance=0.816, threshold=0.680
Разные люди: verified=False, distance=0.956, threshold=0.680
VGG-Face: verified=True, distance=0.484
Facenet512: verified=False, distance=0.365
ArcFace: verified=False, distance=0.816
SFace: verified=False, distance=0.801


## 3. facenet-pytorch

**GitHub:** [timesler/facenet-pytorch](https://github.com/timesler/facenet-pytorch)

### Возможности и преимущества

facenet-pytorch — минималистичная библиотека с двумя основными компонентами: предобученный MTCNN для
детекции лиц (тот же детектор, что мы использовали в Task 3 нашего проекта) и предобученная
Inception-ResNet-V1 (архитектура FaceNet) для извлечения эмбеддингов, с весами, обученными на VGGFace2
или CASIA-Webface. Библиотека полностью на PyTorch — проще всего интегрировать в уже существующий
PyTorch-пайплайн, как раз наш случай.

### Зоопарк моделей

Заметно меньше, чем у первых двух — по сути один backbone (InceptionResnetV1) с выбором из двух
источников предобучения (`vggface2` или `casia-webface`), плюс MTCNN для детекции. Никаких вариантов
ArcFace/margin-based лоссов внутри самой библиотеки нет — модель обучена на классическом
softmax+triplet-подходе оригинальной статьи FaceNet.

### Сложности установки

Самая простая среди трёх для PyTorch-проекта — `pip install facenet-pytorch`, без дополнительных
системных зависимостей (мы использовали именно её для MTCNN-детектора в Task 3). Единственный нюанс —
первая загрузка весов модели идёт с публичного облака при первом вызове, нужен доступ в интернет.

### Требования к ресурсам

Минимальные — MTCNN и InceptionResnetV1 вместе занимают несколько сотен МБ VRAM, свободно работают и
на CPU для единичных фото (хотя заметно медленнее, чем на GPU для батчей).

### Личное мнение

Для проекта, где остальной код уже на чистом PyTorch (как наш) — facenet-pytorch логичнее всего
интегрируется, именно поэтому мы выбрали её для детектора в Task 3. Но для задачи распознавания
(а не только детекции) она заметно уступает по зоопарку моделей InsightFace/DeepFace — если нужна
гибкость в выборе backbone/loss, стоит смотреть на них.


In [6]:
!pip install -q --no-deps facenet-pytorch

from facenet_pytorch import MTCNN, InceptionResnetV1
from PIL import Image
import torch
import torch.nn.functional as F

device = 'cuda' if torch.cuda.is_available() else 'cpu'
mtcnn = MTCNN(image_size=160, margin=0, device=device)
resnet = InceptionResnetV1(pretrained='vggface2').eval().to(device)

img1 = Image.open(image_path_1)
img2 = Image.open(image_path_2)  # тот же человек
img3 = Image.open(image_path_3)  # другой человек

face1 = mtcnn(img1)
face2 = mtcnn(img2)
face3 = mtcnn(img3)

if face1 is not None and face2 is not None and face3 is not None:
    with torch.no_grad():
        emb1 = resnet(face1.unsqueeze(0).to(device))
        emb2 = resnet(face2.unsqueeze(0).to(device))
        emb3 = resnet(face3.unsqueeze(0).to(device))
    sim_same = F.cosine_similarity(emb1, emb2).item()
    sim_diff = F.cosine_similarity(emb1, emb3).item()
    print(f'facenet-pytorch cosine similarity (тот же человек): {sim_same:.3f}')
    print(f'facenet-pytorch cosine similarity (другой человек): {sim_diff:.3f}')
else:
    print('Лицо не найдено на одном из фото')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 2.3 MB/s eta 0:00:0000:0100:010m


  0%|          | 0.00/107M [00:00<?, ?B/s]

facenet-pytorch cosine similarity (тот же человек): 0.593
facenet-pytorch cosine similarity (другой человек): -0.082


## Результаты на собственных тестовых фото

| Библиотека / модель | Тот же человек | Разные люди | Правильно распознала |
|---|---|---|---|
| Наша ArcFace-модель (Task 3) | cos_sim = 0.803 | cos_sim = −0.023 | ✅ |
| InsightFace (buffalo_l) | cos_sim = 0.585 | cos_sim = 0.080 | ✅ |
| facenet-pytorch | cos_sim = 0.593 | cos_sim = −0.082 | ✅ |
| DeepFace / VGG-Face | distance = 0.484 (< threshold 0.68) | — | ✅ |
| DeepFace / ArcFace | distance = 0.816 (> threshold 0.68) | distance = 0.956 | ❌ |
| DeepFace / Facenet512 | distance = 0.365 (verified=False) | — | ❌ |
| DeepFace / SFace | distance = 0.801 (verified=False) | — | ❌ |

Три независимых подхода — наша собственная модель, InsightFace и facenet-pytorch — согласованно и
уверенно отличают одного человека от другого, при этом наша модель показала наибольший разрыв между
парами (0.803 против −0.023). DeepFace дала смешанный результат: три из четырёх встроенных моделей
(включая её собственную реализацию ArcFace) не подтвердили совпадение личности на паре "тот же
человек", кроме VGG-Face. Вероятная причина — DeepFace по умолчанию заново запускает детекцию и
alignment лица *внутри* уже переданного изображения; наши тестовые фото — это тесные bbox-кропы с
запасом 0.35 вокруг лица (без широкого контекста исходного кадра), и встроенный детектор DeepFace,
по всей видимости, не справляется с таким форматом входа так же хорошо, как с обычными "сырыми"
фотографиями. Это практическое наблюдение: разные библиотеки закладывают разные ожидания по
препроцессингу входных изображений, и слепое использование чужого пайплайна без адаптации формата
входа может давать заниженное качество — тот же урок, что мы уже усвоили в Task 3 при настройке
margin вокруг bbox для собственного align'а.